In [1]:
from probabilistic_word_embeddings.estimation import map_estimate 
from probabilistic_word_embeddings.embeddings import Embedding
from simulate import  load_simulated_data_generator, load
import tensorflow as tf

import numpy as np

def sigmoid(x:float): # might be a good idea to have this in a common library.
  return 1 / (1 + np.exp(-x))


def sufficient_statistics(data, vocabulary):
    V = len(vocabulary)
    count_matrix_x1 = np.zeros((V, V), dtype=int)
    count_matrix_total = np.zeros((V, V), dtype=int)

    for item in data:
        v_index = vocabulary[item['v']]
        w_index = vocabulary[item['w']]
        if item['x'] == 1:
            count_matrix_x1[v_index, w_index] += 1
        #elif item['x'] == 0:
        #    count_matrix_x0[v_index, w_index] += 1
        count_matrix_total[v_index, w_index] += 1

    return count_matrix_x1, count_matrix_total

def calculate_p(vi, wi, theta):
    rho_v = theta[vi, :]          # $\rho_v = \theta_v$
    alpha_w = theta[wi + V, :]    # $\alpha_w = \theta_{V + w}$
    eta = rho_v.dot(alpha_w)      # $\eta = \rho_v^T \alpha_w$
    p = sigmoid(eta)              # Run it through the link function $\sigma; p = \sigma(\eta)$
    return p

def compare_theta_to_sufficent_statistics(v, w, theta, counts_1, counts_total, vocabulary):
    V=  len(vocabulary)

    if v == w:
        return None, None
    vi, wi = vocabulary[v], vocabulary[w]
    
    
    rho_v = theta[vi, :]          # $\rho_v = \theta_v$
    alpha_w = theta[wi + V, :]    # $\alpha_w = \theta_{V + w}$
    eta = rho_v.dot(alpha_w)      # $\eta = \rho_v^T \alpha_w$
    p = sigmoid(eta)              # Run it through the link function $\sigma; p = \sigma(\eta)$


    observed_ratio = counts_1[vi, wi] / counts_total[vi, wi] if counts_total[vi, wi] > 0 else 0.0
    return (p, observed_ratio)


def compare_thetas(v, w, theta1, theta2, vocabulary):
    V=  len(vocabulary)

    if v == w:
        return None, None
    vi, wi = vocabulary[v], vocabulary[w]
    
    
    p1 = calculate_p(vi, wi, theta1)
    p2 = calculate_p(vi, wi, theta2)

    return (p1, p2)

In [2]:

import os

for i in os.listdir():
    if i.split(".")[-1] == 'json':
        print(i)

test100k.json
test100k_d2_s08.json
test5k_d2_s08.json


In [3]:


file_path = 'test100k_d2_s08.json'
for item in load_simulated_data_generator(file_path, batch_size=5):
    print(item)  
    break

(<tf.Tensor: shape=(5,), dtype=string, numpy=array([b'word4', b'word8', b'word4', b'word7', b'word7'], dtype=object)>, <tf.Tensor: shape=(5,), dtype=string, numpy=array([b'word5', b'word9', b'word7', b'word6', b'word9'], dtype=object)>, <tf.Tensor: shape=(5,), dtype=float64, numpy=array([1., 0., 1., 1., 0.])>)


In [4]:
json_ = load(file_path)
json_['theta']

[[0.27646735365182884, 0.6572945148009267],
 [0.2643496609467097, -1.0425257852834888],
 [0.7242846933384942, 0.35709965789120907],
 [-0.42956258828822813, 0.4648944833570825],
 [0.2916579169488606, 0.2353059973244208],
 [0.022737793052637432, 0.4373703892899576],
 [-0.5891632696013336, -0.13032795839444222],
 [-0.38569545014398265, 0.47907697010770206],
 [0.031777685985327195, -0.2339654007720709],
 [-0.6255267698854737, -0.20575379249509657],
 [0.006513744414674807, -0.22048232423949635],
 [1.0352510515185658, 0.8053794522446355],
 [-2.168929983172775, -1.5112105967741383],
 [-0.13981767364412956, -0.3377523292610829],
 [0.17091439799888888, 0.17385754481805088],
 [1.6942710040408386, -0.8896166101538251],
 [-0.3020840057015985, 1.6342172859938644],
 [0.5173623969614776, 0.5304506979010094],
 [-0.41120509734997035, -1.3184601366845223],
 [0.1339717953781929, 0.08721127025723803]]

In [5]:
counts_1, counts_total = sufficient_statistics(json_['data'],json_['vocabulary'])

display(counts_1)

display(counts_total)

array([[  0, 725, 190, 474, 635, 517, 784, 698, 302, 581],
       [617,   0, 805, 658, 474, 901, 159, 407, 901, 549],
       [568, 868,   0, 527, 578, 812, 669, 719, 345, 593],
       [502, 527, 621,   0, 550, 287, 765, 551, 419, 539],
       [525, 709, 327, 556,   0, 674, 613, 632, 423, 601],
       [499, 674, 373, 541, 612,   0, 668, 632, 416, 558],
       [569, 379, 899, 584, 567, 337,   0, 466, 671, 503],
       [529, 583, 587, 509, 551, 293, 783,   0, 422, 559],
       [542, 537, 660, 536, 481, 652, 467, 532,   0, 538],
       [534, 375, 907, 580, 504, 312, 514, 397, 704,   0]])

array([[   0, 1057, 1108, 1096, 1129, 1131, 1091, 1135, 1119, 1142],
       [1115,    0, 1121, 1149, 1058, 1137, 1159, 1079, 1157, 1114],
       [1160, 1174,    0, 1144, 1052, 1132, 1121, 1119, 1069, 1114],
       [1074, 1092, 1120,    0, 1097, 1114, 1070, 1142, 1078, 1082],
       [1127, 1136, 1156, 1132,    0, 1122, 1096, 1131, 1105, 1187],
       [1035, 1158, 1102, 1159, 1125,    0, 1047, 1089, 1122, 1127],
       [1129, 1125, 1090, 1111, 1187, 1083,    0, 1128, 1108, 1041],
       [1052, 1155, 1137, 1103, 1098, 1139, 1102,    0, 1089, 1132],
       [1045, 1178, 1135, 1072, 1016, 1124, 1117, 1110,    0, 1139],
       [1041, 1165, 1073, 1082, 1110, 1133, 1079, 1041, 1124,    0]])

In [8]:
vocabulary = json_['vocabulary']
counts_1, counts_total = sufficient_statistics(json_['data'],json_['vocabulary'])

words = list(vocabulary.keys())
theta = np.array(json_['theta'])

store_p = []
store_observer_ratios = []
for v in words:
    for w in words:
        p, observed_ratio = compare_theta_to_sufficent_statistics(v, w, theta, counts_1, counts_total, vocabulary)

        if p is None:
            pass
        else:
            print(f'{v}, {w}')
            print('from theta :', p)
            print('observed   :', observed_ratio)
            print()
            store_p.append(p)
            store_observer_ratios.append(observed_ratio)


print('CORRELATION:', np.corrcoef(store_p, store_observer_ratios)[1,0])


word0, word1
from theta : 0.6932982734095837
observed   : 0.6859035004730369

word0, word2
from theta : 0.16896943021384314
observed   : 0.17148014440433212

word0, word3
from theta : 0.43520201926250096
observed   : 0.4324817518248175

word0, word4
from theta : 0.5402943926215014
observed   : 0.562444641275465

word0, word5
from theta : 0.4709503779218886
observed   : 0.45711759504862953

word0, word6
from theta : 0.7292154398291375
observed   : 0.7186067827681026

word0, word7
from theta : 0.6205059197217694
observed   : 0.614977973568282

word0, word8
from theta : 0.27283198322631497
observed   : 0.26988382484361034

word0, word9
from theta : 0.5235730902025684
observed   : 0.5087565674255692

word1, word0
from theta : 0.5576377433101215
observed   : 0.5533632286995516

word1, word2
from theta : 0.7314752130009382
observed   : 0.7181088314005353

word1, word3
from theta : 0.578142977758067
observed   : 0.5726718885987816

word1, word4
from theta : 0.4660349365099926
observed   : 0.4

In [9]:

N = 100000
# ln(2) = 0.6931471805599453
#python simulate.py --save_path "test100k.json" --vocab_size 10 --dimensionality 3 --n_datapoints 100000 --seed 1 --sd 0.5
e = Embedding(vocabulary=set(json_['vocabulary'].keys()), dimensionality=2, lambda0=0.0)


theta_estimated_before = e[list(vocabulary.keys()) + [w + '_c' for w in list(vocabulary.keys())]].numpy()
magnitude_before =  tf.reduce_mean(tf.math.abs(theta_estimated_before))
print('Record magnitude before:' , magnitude_before)

estimate = map_estimate(e, data_generator=load_simulated_data_generator(file_path, batch_size=100), batch_size=100, N = N, evaluate=False, training_loss=True, epochs = 10)

Record magnitude before: tf.Tensor(0.13385443024087276, shape=(), dtype=float64)
10:29:25 [TRAIN] (map): Epoch 0


c:\Users\isacb\AppData\Local\Programs\Python\Python39\lib\site-packages\probabilistic_word_embeddings\estimation.py:56: UserWarning: The arguments 'ws' and 'ns' are ignored when using 'data_generator'
  warnings.warn("The arguments 'ws' and 'ns' are ignored when using 'data_generator'")
  0% (0 of 1000) |                       | Elapsed Time: 0:00:00 ETA:  --:--:--
  0% (1 of 1000) |                       | Elapsed Time: 0:00:00 ETA:   0:03:26
  1% (13 of 1000) |                      | Elapsed Time: 0:00:00 ETA:   0:03:24
  2% (26 of 1000) |                      | Elapsed Time: 0:00:00 ETA:   0:00:13
  3% (38 of 1000) |                      | Elapsed Time: 0:00:00 ETA:   0:00:12
  5% (51 of 1000) |#                     | Elapsed Time: 0:00:00 ETA:   0:00:09
  6% (64 of 1000) |#                     | Elapsed Time: 0:00:00 ETA:   0:00:08
  7% (76 of 1000) |#                     | Elapsed Time: 0:00:00 ETA:   0:00:07
  8% (89 of 1000) |#                     | Elapsed Time: 0:00:00 ETA:   

10:29:27 [TRAIN] (map): Epoch 0 mean training loss after 250 batches: 0.7180522897546117


 25% (254 of 1000) |#####                | Elapsed Time: 0:00:01 ETA:   0:00:04
 26% (266 of 1000) |#####                | Elapsed Time: 0:00:01 ETA:   0:00:04
 27% (279 of 1000) |#####                | Elapsed Time: 0:00:01 ETA:   0:00:04
 29% (292 of 1000) |######               | Elapsed Time: 0:00:01 ETA:   0:00:04
 30% (304 of 1000) |######               | Elapsed Time: 0:00:01 ETA:   0:00:04
 31% (317 of 1000) |######               | Elapsed Time: 0:00:02 ETA:   0:00:04
 33% (330 of 1000) |######               | Elapsed Time: 0:00:02 ETA:   0:00:04
 34% (342 of 1000) |#######              | Elapsed Time: 0:00:02 ETA:   0:00:04
 35% (355 of 1000) |#######              | Elapsed Time: 0:00:02 ETA:   0:00:03
 36% (368 of 1000) |#######              | Elapsed Time: 0:00:02 ETA:   0:00:03
 38% (380 of 1000) |#######              | Elapsed Time: 0:00:02 ETA:   0:00:03
 39% (393 of 1000) |########             | Elapsed Time: 0:00:02 ETA:   0:00:03
 40% (406 of 1000) |########            

10:29:28 [TRAIN] (map): Epoch 0 mean training loss after 500 batches: 0.7091489128233435


...

 50% (505 of 1000) |##########           | Elapsed Time: 0:00:04 ETA:   0:00:04
 51% (514 of 1000) |##########           | Elapsed Time: 0:00:05 ETA:   0:00:04
 51% (519 of 1000) |##########           | Elapsed Time: 0:00:05 ETA:   0:00:04
 52% (529 of 1000) |###########          | Elapsed Time: 0:00:05 ETA:   0:00:04
 53% (539 of 1000) |###########          | Elapsed Time: 0:00:05 ETA:   0:00:04
 54% (545 of 1000) |###########          | Elapsed Time: 0:00:05 ETA:   0:00:04
 55% (555 of 1000) |###########          | Elapsed Time: 0:00:05 ETA:   0:00:04
 56% (565 of 1000) |###########          | Elapsed Time: 0:00:05 ETA:   0:00:04
 57% (570 of 1000) |###########          | Elapsed Time: 0:00:05 ETA:   0:00:04
 58% (580 of 1000) |############         | Elapsed Time: 0:00:05 ETA:   0:00:04
 59% (591 of 1000) |############         | Elapsed Time: 0:00:05 ETA:   0:00:04
 60% (601 of 1000) |############         | Elapsed Time: 0:00:05 ETA:   0:00:04
 60% (608 of 1000) |############        

10:30:50 [TRAIN] (map): Epoch 9 mean training loss after 750 batches: 0.6920680344822543


 75% (754 of 1000) |###############      | Elapsed Time: 0:00:07 ETA:   0:00:02
 76% (760 of 1000) |###############      | Elapsed Time: 0:00:07 ETA:   0:00:02
 77% (770 of 1000) |################     | Elapsed Time: 0:00:07 ETA:   0:00:02
 78% (780 of 1000) |################     | Elapsed Time: 0:00:07 ETA:   0:00:02
 78% (785 of 1000) |################     | Elapsed Time: 0:00:07 ETA:   0:00:02
 79% (796 of 1000) |################     | Elapsed Time: 0:00:07 ETA:   0:00:01
 80% (807 of 1000) |################     | Elapsed Time: 0:00:07 ETA:   0:00:01
 81% (818 of 1000) |#################    | Elapsed Time: 0:00:08 ETA:   0:00:01
 82% (823 of 1000) |#################    | Elapsed Time: 0:00:08 ETA:   0:00:01
 83% (834 of 1000) |#################    | Elapsed Time: 0:00:08 ETA:   0:00:01
 84% (845 of 1000) |#################    | Elapsed Time: 0:00:08 ETA:   0:00:01
 85% (856 of 1000) |#################    | Elapsed Time: 0:00:08 ETA:   0:00:01
 86% (867 of 1000) |##################  

10:30:53 [TRAIN] (map): Epoch 9 mean training loss after 1000 batches: 0.6920339142706049


100% (1000 of 1000) |####################| Elapsed Time: 0:00:09 Time:  0:00:09


In [11]:
vocabulary = json_['vocabulary']
counts_1, counts_total = sufficient_statistics(json_['data'],json_['vocabulary'])

words = list(vocabulary.keys())
#theta = np.array(json_['theta'])

theta_estimated = e[list(vocabulary.keys()) + [w + '_c' for w in list(vocabulary.keys())]].numpy()


store_p = []
store_p_estimated = []
for v in words:
    for w in words:
        p, p_estimated = compare_thetas(v, w, theta, theta_estimated, vocabulary)

        if p is None:
            pass
        else:
            print(f'{v}, {w}')
            print('from theta :', p)
            print('observed   :', p_estimated)
            print()
            store_p.append(p)
            store_p_estimated.append(p_estimated)


print('CORRELATION:', np.corrcoef(store_p, store_p_estimated)[1,0])


word0, word1
from theta : 0.6932982734095837
observed   : 0.49949111503520394

word0, word2
from theta : 0.16896943021384314
observed   : 0.49921535970169884

word0, word3
from theta : 0.43520201926250096
observed   : 0.5004644922303557

word0, word4
from theta : 0.5402943926215014
observed   : 0.5010948540599662

word0, word5
from theta : 0.4709503779218886
observed   : 0.5015365771039945

word0, word6
from theta : 0.7292154398291375
observed   : 0.4987376520116101

word0, word7
from theta : 0.6205059197217694
observed   : 0.49980589813768966

word0, word8
from theta : 0.27283198322631497
observed   : 0.49991699474188067

word0, word9
from theta : 0.5235730902025684
observed   : 0.5011070441860216

word1, word0
from theta : 0.5576377433101215
observed   : 0.5008559156844139

word1, word2
from theta : 0.7314752130009382
observed   : 0.5032341266674629

word1, word3
from theta : 0.578142977758067
observed   : 0.500753823236182

word1, word4
from theta : 0.4660349365099926
observed   : 0

In [12]:
print('Magnitude before:',  tf.reduce_mean(tf.math.abs(theta_estimated_before)))
theta_estimated = e[list(vocabulary.keys()) + [w + '_c' for w in list(vocabulary.keys())]].numpy()
print('Magnitude after :', tf.reduce_mean(tf.math.abs(theta_estimated)))




Magnitude before: tf.Tensor(0.13385443024087276, shape=(), dtype=float64)
Magnitude after : tf.Tensor(0.07319613707306209, shape=(), dtype=float64)


In [14]:
tf.__version__

'2.11.0'